# 03 — Full-year one-arm profile for the department desktop

This version records **current RSS and cumulative peak RSS** throughout the
full pipeline, while retaining the same numerical operator and timing
configuration. Run it from a freshly restarted kernel.


In [1]:
import gc
import os
import platform
import resource
import sys

# This notebook benchmarks the CPU implementation.
# These must be set before importing JAX, FEW, or packages that import them.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import jax.numpy as jnp

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform

import json
import platform
import subprocess
import time
from pathlib import Path

import jax
import numpy as np

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

import WDM
import WDM_LISAresponse
from WDM.code.time_delay_filters import _time_shift_jax as time_shift_jax


process = psutil.Process(os.getpid()) if psutil is not None else None
memory_stages: dict[str, dict[str, float | int | None]] = {}


def current_rss_bytes():
    """Current resident set size, including native/JAX allocations."""
    return (
        int(process.memory_info().rss)
        if process is not None
        else None
    )


def peak_rss_bytes():
    """Process high-water RSS. Linux reports ru_maxrss in KiB."""
    value = int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    if sys.platform == "darwin":
        return value
    return value * 1024


def record_memory(stage: str):
    """Record current and cumulative peak RSS at a named pipeline stage."""
    rss = current_rss_bytes()
    peak = peak_rss_bytes()
    snapshot = {
        "rss_bytes": rss,
        "peak_rss_bytes": peak,
        "rss_gib": rss / 1024**3 if rss is not None else None,
        "peak_rss_gib": peak / 1024**3,
    }
    memory_stages[stage] = snapshot
    rss_text = (
        f"{snapshot['rss_gib']:.3f} GiB"
        if snapshot["rss_gib"] is not None
        else "unavailable"
    )
    print(
        f"Memory [{stage}]: "
        f"RSS={rss_text} | "
        f"peak={snapshot['peak_rss_gib']:.3f} GiB"
    )
    return snapshot


def rss_gib():
    rss = current_rss_bytes()
    return rss / 1024**3 if rss is not None else float("nan")


record_memory("after_imports")

NOTEBOOK_DIR = Path.cwd().resolve()
RESPONSE_ROOT = next(
    (
        path for path in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
        if (path / "examples").exists()
        and ((path / "src" / "WDM_LISAresponse").exists()
             or (path / "WDM_LISAresponse").exists())
    ),
    None,
)
if RESPONSE_ROOT is None:
    # Installed-package fallback.
    RESPONSE_ROOT = Path(WDM_LISAresponse.__file__).resolve().parents[2]

ORBIT_FILE = RESPONSE_ROOT / "examples" / "esa-trailing-orbits.h5"
if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Could not find the orbit file at "
        f"{ORBIT_FILE}. The file should remain in WDM_LISAresponse/examples."
    )


def load_lisa_orbits(path: Path):
    """Load the physical LISA orbit file used by the response calculations."""
    orbit_data = read_orbit_file(str(path))
    orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
    orbits = InterpolatedOrbits(
        orbit_times,
        orbit_data.positions,
        orbit_data.velocities,
    )
    return orbit_data, orbits


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


def json_ready(value):
    """Convert NumPy and JAX objects into JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item") and getattr(value, "ndim", None) == 0:
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def save_json(filename: str, payload) -> Path:
    """Save a JSON result beside the notebook."""
    path = NOTEBOOK_DIR / filename
    path.write_text(
        json.dumps(json_ready(payload), indent=2),
        encoding="utf-8",
    )
    print("Saved:", path)
    return path


orbit_data, orbits = load_lisa_orbits(ORBIT_FILE)
record_memory("after_orbits")
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Python:", platform.python_version())
print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("JAX devices:", jax.devices())
if jax.default_backend() != "cpu":
    raise RuntimeError("Notebook 03 must run on the CPU.")
wdm_execution_representation = getattr(
    time_shift_jax,
    "CPU_REAL_EXECUTION_REPRESENTATION",
    None,
)
wdm_cpu_complex_dispatch_active = (
    wdm_execution_representation == "complex"
    and time_shift_jax._array_uses_cpu_backend(jnp.zeros(1))
)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("WDM_LISAresponse source:", git_state(WDM_LISAresponse.__file__))
print("WDM:", WDM.__file__)
print("WDM source:", git_state(WDM.__file__))
print("WDM CPU real execution:", wdm_execution_representation)
print("WDM CPU complex dispatch active:", wdm_cpu_complex_dispatch_active)
if not wdm_cpu_complex_dispatch_active:
    raise RuntimeError(
        "The imported WDM checkout does not expose the optimized CPU "
        "real-via-complex execution path. Install the patched WDM checkout "
        "and restart the notebook kernel."
    )


from WDM_LISAresponse.code.lisa_response.one_arm_plan import OneArmWDMStaticPlan
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_context_from_wdm,
    _build_wdm_dT_grid,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

def block_result(value):
    """Synchronise every JAX leaf before stopping a timer."""
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def timed(function):
    started = time.perf_counter()
    value = function()
    block_result(value)
    return value, float(time.perf_counter() - started)


def timing_summary(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "all_s": [float(value) for value in values],
    }


def benchmark(function, *, warmups: int, repeats: int):
    for _ in range(int(warmups)):
        block_result(function())

    samples = []
    latest = None
    for _ in range(int(repeats)):
        latest, elapsed = timed(function)
        samples.append(elapsed)
    return latest, timing_summary(samples)


def compare_arrays(candidate, reference, *, m_start: int = 0):
    """Return max-absolute and relative-L2 differences."""
    candidate = jax.numpy.asarray(candidate)[..., m_start:]
    reference = jax.numpy.asarray(reference)[..., m_start:]
    difference = candidate - reference

    difference_power = jax.numpy.sum(jax.numpy.abs(difference) ** 2)
    reference_power = jax.numpy.sum(jax.numpy.abs(reference) ** 2)
    max_abs = jax.numpy.max(jax.numpy.abs(difference))
    rel_l2 = jax.numpy.where(
        reference_power > 0,
        jax.numpy.sqrt(difference_power / reference_power),
        jax.numpy.sqrt(difference_power),
    )
    block_result((max_abs, rel_l2))
    return {"max_abs": float(max_abs), "rel_l2": float(rel_l2)}


def compare_link_maps(candidate, reference, *, m_start: int = 0):
    if set(candidate) != set(reference):
        raise ValueError("Directed-link dictionaries contain different keys.")

    metrics = {
        str(link): compare_arrays(
            candidate[link],
            reference[link],
            m_start=m_start,
        )
        for link in reference
    }
    return {
        "per_link": metrics,
        "max_abs": max(item["max_abs"] for item in metrics.values()),
        "rel_l2": max(item["rel_l2"] for item in metrics.values()),
    }


print("Machine:", platform.node())
print("CPU:", platform.processor())
print(
    "Physical cores:",
    psutil.cpu_count(logical=False) if psutil is not None else "unknown",
)


Memory [after_imports]: RSS=0.517 GiB | peak=0.516 GiB
Memory [after_orbits]: RSS=0.520 GiB | peak=0.519 GiB
Notebook directory: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples
Orbit file: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/esa-trailing-orbits.h5
Python: 3.12.12
JAX: 0.9.2 | backend: cpu
JAX devices: [CpuDevice(id=0)]
WDM_LISAresponse: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
WDM_LISAresponse source: {'root': '/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse', 'commit': 'd39df69cacfcdb9e7aa5a5179de25af4a82b0574', 'dirty': False}
WDM: /data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py
WDM source: {'root': '/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets', 'commit': 'a32028bf6d8d9ffe0cc6a76b45622d34679df34c', 'dirty': False}
WDM CPU real execution: complex
WDM CPU complex dispatch active: True
Machine: calx038.ast.cam.ac.uk
CPU: x86_64
Physical cores: 20


In [2]:
# Full-year production configuration

N = 3_200_000
dt = 10.0
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False

ROW_CHUNK_SIZE = 32_640
LAG_BLOCK_SIZE = 17

SHIFT_KWARGS = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": ROW_CHUNK_SIZE,
    "lag_block_size": LAG_BLOCK_SIZE,
    "batch_chunk": 6,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

theta = np.pi / 2
phi = 0.0

DWT_WARMUPS = 2
DWT_REPEATS = 11
SHIFT_WARMUPS = 2
SHIFT_REPEATS = 11
PIPELINE_WARMUPS = 2
PIPELINE_REPEATS = 9

PARALLEL_SCHEDULES = [
    {"name": "groups_3_workers_2", "group_size": 3, "workers": 2},
    {"name": "groups_2_workers_3", "group_size": 2, "workers": 3},
    {"name": "groups_1_workers_6", "group_size": 1, "workers": 6},
]

SAVE_RESULTS = True
REUSE_CACHE = True


## 1. Local cache and physical inputs

The raw one-year waveform and WDM coefficients are large and may depend on
FEW, WDM and JAX versions. They are cached beside the notebook with
`_local_wdm_` filenames.

Delete the relevant cache files, or set `REUSE_CACHE=False`, after changing
the waveform generator, WDM transform, `calc_m0`, sampling or tiling.

In [3]:

source_state = git_state(WDM_LISAresponse.__file__)
wdm_source_state = git_state(WDM.__file__)

waveform_cache = NOTEBOOK_DIR / "_local_wdm_03_waveform_cache.npz"
wdm_cache = NOTEBOOK_DIR / "_local_wdm_03_wdm_coefficients.npz"
cache_metadata = NOTEBOOK_DIR / "_local_wdm_03_cache_metadata.json"

# `orbit_data`, `orbits` and `t0` were loaded in the setup cell.
td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs={"pad_output": True, "odd_len": True},
    return_list=True,
    force_backend="cpu",
)

model_kwargs = {
    "td_gen": td_gen,
    "m1": 1.0e6,
    "m2": 10.0,
    "a": 0.9,
    "p0": 8.5,
    "e0": 0.3,
    "x0": 1.0,
    "dist": 1.0,
    "theta": theta,
    "phi": phi,
    "qK": np.pi / 3,
    "phiK": np.pi / 3,
    "Phi_phi0": np.pi / 3,
    "Phi_theta0": 0.0,
    "Phi_r0": np.pi / 3,
    "eps": 1.0e-2,
}

if REUSE_CACHE and waveform_cache.exists():
    with np.load(waveform_cache, allow_pickle=False) as cached:
        times_pad = np.asarray(cached["times_pad"])
        h_plus = np.asarray(cached["h_plus"])
        h_cross = np.asarray(cached["h_cross"])
    waveform_generation_s = 0.0
    print("Reused waveform cache:", waveform_cache)
else:
    started = time.perf_counter()
    times_pad, h_plus, h_cross, padding = (
        generate_windowed_padded_waveform(
            waveform_model="emri",
            t0=t0,
            dt=dt,
            N=N,
            model_kwargs=model_kwargs,
            window_function="tukey",
            window_kwargs={"alpha": 0.005, "tukey_factor": 4.0},
            wdm_Nf=Nf,
            return_padding_info=True,
        )
    )
    waveform_generation_s = time.perf_counter() - started
    np.savez(
        waveform_cache,
        times_pad=times_pad,
        h_plus=h_plus,
        h_cross=h_cross,
    )
    print("Created waveform cache:", waveform_cache)

print("Padded samples:", times_pad.size)
print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
record_memory("after_waveform")


Created waveform cache: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_waveform_cache.npz
Padded samples: 3264000
Waveform generation_s: 8.789
Memory [after_waveform]: RSS=6.558 GiB | peak=6.651 GiB


{'rss_bytes': 7041077248,
 'peak_rss_bytes': 7141408768,
 'rss_gib': 6.557514190673828,
 'peak_rss_gib': 6.6509552001953125}

In [4]:
# Build the WDM grid and reusable static one-arm plan.

grid_context, grid_s = timed(
    lambda: _build_wdm_dT_grid(
        times_pad,
        Nf=Nf,
        d=d_wdm,
        A_frac=A_frac,
        q=q_wdm,
        calc_m0=CALC_M0,
    )
)
record_memory("after_grid")

one_arm_plan, plan_s = timed(
    lambda: OneArmWDMStaticPlan.build_from_context(
        grid_context,
        orbits,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=SHIFT_KWARGS,
    )
)
record_memory("after_plan")

print("WDM shape:", (grid_context["wdm"].Nt, grid_context["wdm"].Nf))
print("Grid build_s:", f"{grid_s:.3f}")
print("Static plan build_s:", f"{plan_s:.3f}")
print(
    "Persistent interpolation-table memory [MiB]:",
    one_arm_plan.kernel_memory_bytes / 1024**2,
)


Memory [after_grid]: RSS=6.972 GiB | peak=7.297 GiB
Memory [after_plan]: RSS=7.155 GiB | peak=8.186 GiB
WDM shape: (32640, 100)
Grid build_s: 2.892
Plan build_s: 1.086
Persistent plan memory [MiB]: 153.14960861206055


## 2. WDM transforms and reusable local coefficient cache

In [5]:
def sequential_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=False,
    )


def parallel_dwt():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        h_plus,
        h_cross,
        grid_context["wdm"],
        parallel_wdm_transforms=True,
    )


sequential_context, sequential_dwt_stats = benchmark(
    sequential_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)
parallel_context, parallel_dwt_stats = benchmark(
    parallel_dwt,
    warmups=DWT_WARMUPS,
    repeats=DWT_REPEATS,
)
record_memory("after_dwt_benchmarks")

dwt_plus = compare_arrays(
    parallel_context["w_t_plus"],
    sequential_context["w_t_plus"],
    m_start=1,
)
dwt_cross = compare_arrays(
    parallel_context["w_t_cross"],
    sequential_context["w_t_cross"],
    m_start=1,
)

# Preserve the real coefficient dtype used by the cleaned production path.
w_plus_host = np.asarray(parallel_context["w_t_plus"])
w_cross_host = np.asarray(parallel_context["w_t_cross"])

np.savez(
    wdm_cache,
    times_pad=times_pad,
    w_plus=w_plus_host,
    w_cross=w_cross_host,
    Nf=np.asarray(Nf),
    q_wdm=np.asarray(q_wdm),
    d_wdm=np.asarray(d_wdm),
    A_frac=np.asarray(A_frac),
    calc_m0=np.asarray(CALC_M0),
)
cache_metadata.write_text(
    json.dumps(
        json_ready(
            {
                "source": source_state,
                "configuration": {
                    "N": N,
                    "dt": dt,
                    "Nf": Nf,
                    "q_wdm": q_wdm,
                    "d_wdm": d_wdm,
                    "A_frac": A_frac,
                    "calc_m0": CALC_M0,
                },
                "shape": list(w_plus_host.shape),
                "dtype": str(w_plus_host.dtype),
            }
        ),
        indent=2,
    ),
    encoding="utf-8",
)

print("Sequential DWT:", sequential_dwt_stats)
print("Parallel DWT:", parallel_dwt_stats)
print("DWT comparisons:", dwt_plus, dwt_cross)
print("Saved WDM coefficient cache:", wdm_cache)
record_memory("after_wdm_cache")


Memory [after_dwt_benchmarks]: RSS=7.787 GiB | peak=8.186 GiB
Sequential DWT: {'median_s': 0.2026010180125013, 'q25_s': 0.198681043490069, 'q75_s': 0.20536004399764352, 'iqr_s': 0.0066790005075745285, 'minimum_s': 0.19496985198929906, 'maximum_s': 0.21054927801014856, 'all_s': [0.20385260396869853, 0.2026010180125013, 0.21054927801014856, 0.20511162199545652, 0.1981578579870984, 0.19920422899303958, 0.19496985198929906, 0.2023808480007574, 0.20805826695868745, 0.1969083550502546, 0.20560846599983051]}
Parallel DWT: {'median_s': 0.12419528502505273, 'q25_s': 0.12242254850571044, 'q75_s': 0.1260428609966766, 'iqr_s': 0.003620312490966171, 'minimum_s': 0.12096597795607522, 'maximum_s': 0.12856107100378722, 'all_s': [0.12338619196088985, 0.12173203501151875, 0.1259910449734889, 0.12544380000326782, 0.12856107100378722, 0.1275783510063775, 0.12165593303507194, 0.12311306199990213, 0.12419528502505273, 0.12096597795607522, 0.12609467701986432]}
DWT comparisons: {'max_abs': 0.0, 'rel_l2': 0.0

{'rss_bytes': 8361930752,
 'peak_rss_bytes': 8789245952,
 'rss_gib': 7.787654876708984,
 'peak_rss_gib': 8.185623168945312}

## 3. Six-link scheduling sweep


In [6]:
w_plus = jnp.asarray(w_plus_host)
w_cross = jnp.asarray(w_cross_host)
block_result((w_plus, w_cross))
record_memory("after_device_coefficients")

def runtime_sky_shift():
    return one_arm_plan.apply_device(
        w_plus,
        w_cross,
        theta,
        phi,
    )

baseline_links, baseline_shift_stats = benchmark(
    runtime_sky_shift,
    warmups=SHIFT_WARMUPS,
    repeats=SHIFT_REPEATS,
)
record_memory("after_runtime_sky_benchmark")

schedule_rows = [
    {
        "name": "runtime_sky",
        **baseline_shift_stats,
    }
]
best_schedule = schedule_rows[0]

print("Runtime sky + one-arm:", baseline_shift_stats)


Memory [after_device_coefficients]: RSS=7.788 GiB | peak=8.186 GiB
Memory [after_materialized_shift_benchmark]: RSS=8.336 GiB | peak=8.798 GiB
Memory [after_schedule_sweep]: RSS=9.396 GiB | peak=10.265 GiB


,name,group_size,workers,median_s,q25_s,q75_s,iqr_s,minimum_s,maximum_s,all_s,max_abs,rel_l2
0,materialized,6,1,0.445145,0.432519,0.457746,0.025227,0.419970,0.463915,"[0.445145316014532, 0.4311871620011516, 0.4639...",0.0,0.0
1,groups_3_workers_2,3,2,0.338994,0.337237,0.340141,0.002903,0.332029,0.348250,"[0.33979286201065406, 0.3352293290081434, 0.33...",0.0,0.0
2,groups_2_workers_3,2,3,0.327625,0.325457,0.330919,0.005462,0.320271,0.341267,"[0.32620489498367533, 0.33629437803756446, 0.3...",0.0,0.0
3,groups_1_workers_6,1,6,0.332750,0.329661,0.338262,0.008600,0.328433,0.349051,"[0.3284327939618379, 0.33274983998853713, 0.33...",0.0,0.0


Selected desktop schedule: groups_2_workers_3


## 4. Complete full-year A1


In [7]:
def baseline_a1():
    context = sequential_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        theta,
        phi,
    )

def selected_a1():
    context = parallel_dwt()
    return one_arm_plan.apply_device(
        context["w_t_plus"],
        context["w_t_cross"],
        theta,
        phi,
    )

baseline_a1_output, baseline_a1_stats = benchmark(
    baseline_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)
selected_a1_output, selected_a1_stats = benchmark(
    selected_a1,
    warmups=PIPELINE_WARMUPS,
    repeats=PIPELINE_REPEATS,
)

a1_comparison = compare_link_maps(
    selected_a1_output,
    baseline_a1_output,
    m_start=1,
)

print("Sequential-DWT runtime:", baseline_a1_stats)
print("Parallel-DWT runtime:", selected_a1_stats)
print("A1 comparison:", a1_comparison)


Memory [after_baseline_a1]: RSS=9.493 GiB | peak=10.265 GiB
Memory [after_selected_a1]: RSS=9.557 GiB | peak=10.534 GiB
Baseline A1: {'median_s': 0.635370907955803, 'q25_s': 0.6325619039707817, 'q75_s': 0.6459270319901407, 'iqr_s': 0.013365128019358963, 'minimum_s': 0.621204114984721, 'maximum_s': 0.6602370360051282, 'all_s': [0.621204114984721, 0.6461976270074956, 0.6459270319901407, 0.6325619039707817, 0.6220888860407285, 0.635370907955803, 0.6602370360051282, 0.6390900840051472, 0.633533363987226]}
Selected A1: {'median_s': 0.45342600101139396, 'q25_s': 0.45014252298278734, 'q75_s': 0.4535488300025463, 'iqr_s': 0.0034063070197589695, 'minimum_s': 0.44724227098049596, 'maximum_s': 0.4689326059888117, 'all_s': [0.4689326059888117, 0.4503715259488672, 0.45342600101139396, 0.4535488300025463, 0.45753125700866804, 0.44880532001843676, 0.44724227098049596, 0.45014252298278734, 0.4534675079630688]}
Comparison: {'per_link': {'(2, 1)': {'max_abs': 0.0, 'rel_l2': 0.0}, '(3, 2)': {'max_abs': 0

{'rss_bytes': 10261237760,
 'peak_rss_bytes': 11310727168,
 'rss_gib': 9.556522369384766,
 'peak_rss_gib': 10.533935546875}

In [8]:
record_memory("final")

results = {
    "source": source_state,
    "dependencies": {
        "WDM": {
            "module_file": str(Path(WDM.__file__).resolve()),
            "source": wdm_source_state,
            "cpu_real_execution_representation": (
                wdm_execution_representation
            ),
            "cpu_complex_dispatch_active": (
                wdm_cpu_complex_dispatch_active
            ),
        },
    },
    "machine": {
        "node": platform.node(),
        "processor": platform.processor(),
        "physical_cores": (
            psutil.cpu_count(logical=False)
            if psutil is not None
            else None
        ),
    },
    "configuration": {
        "N": N,
        "dt": dt,
        "Nf": Nf,
        "L_trunc": L_TRUNC,
        "calc_m0": CALC_M0,
        "shift_kwargs": SHIFT_KWARGS,
    },
    "cache_files": {
        "waveform": str(waveform_cache),
        "wdm_coefficients": str(wdm_cache),
        "metadata": str(cache_metadata),
    },
    "setup_s": {
        "waveform_generation": waveform_generation_s,
        "grid": grid_s,
        "one_arm_static_plan": plan_s,
    },
    "dwt": {
        "sequential": sequential_dwt_stats,
        "parallel": parallel_dwt_stats,
        "plus_comparison": dwt_plus,
        "cross_comparison": dwt_cross,
    },
    "shift_schedules": schedule_rows,
    "selected_schedule": best_schedule,
    "complete_a1": {
        "baseline": baseline_a1_stats,
        "selected": selected_a1_stats,
        "comparison": a1_comparison,
    },
    "memory": {
        "plan_bytes": one_arm_plan.shift_plan.device_plan_memory_bytes,
        "waveform_host_bytes": int(
            np.asarray(times_pad).nbytes
            + np.asarray(h_plus).nbytes
            + np.asarray(h_cross).nbytes
        ),
        "wdm_coefficients_host_bytes": int(
            np.asarray(w_plus_host).nbytes
            + np.asarray(w_cross_host).nbytes
        ),
        "wdm_coefficients_device_bytes": int(
            w_plus.nbytes + w_cross.nbytes
        ),
        "current_rss_gib": rss_gib(),
        "peak_rss_gib": peak_rss_bytes() / 1024**3,
        "stages": memory_stages,
    },
}

print(json.dumps(json_ready(results), indent=2))

memory_rows = [
    {
        "stage": stage,
        "rss_gib": snapshot["rss_gib"],
        "peak_rss_gib": snapshot["peak_rss_gib"],
    }
    for stage, snapshot in memory_stages.items()
]
if pd is not None:
    memory_table = pd.DataFrame(memory_rows)
    display(memory_table)
else:
    for row in memory_rows:
        print(row)


if SAVE_RESULTS:
    save_json("_local_wdm_03_desktop_profile.json", results)
    if pd is not None:
        schedule_path = (
            NOTEBOOK_DIR / "_local_wdm_03_desktop_shift_schedule.csv"
        )
        pd.DataFrame(schedule_rows).to_csv(
            schedule_path,
            index=False,
        )
        print("Saved:", schedule_path)

        memory_path = NOTEBOOK_DIR / "_local_wdm_03_memory_stages.csv"
        pd.DataFrame(memory_rows).to_csv(memory_path, index=False)
        print("Saved:", memory_path)


Memory [final]: RSS=9.557 GiB | peak=10.534 GiB
{
  "source": {
    "root": "/data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse",
    "commit": "d39df69cacfcdb9e7aa5a5179de25af4a82b0574",
    "dirty": false
  },
  "dependencies": {
    "WDM": {
      "module_file": "/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py",
      "source": {
        "root": "/data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets",
        "commit": "a32028bf6d8d9ffe0cc6a76b45622d34679df34c",
        "dirty": false
      },
      "cpu_real_execution_representation": "complex",
      "cpu_complex_dispatch_active": true
    }
  },
  "machine": {
    "node": "calx038.ast.cam.ac.uk",
    "processor": "x86_64",
    "physical_cores": 20
  },
  "configuration": {
    "N": 3200000,
    "dt": 10.0,
    "Nf": 100,
    "L_trunc": 25,
    "calc_m0": false,
    "shift_kwargs": {
      "assembly_backend": "production",
      "assembly_precision": "complex64",
      "row_chunk_size": 32640,
    

,stage,rss_gib,peak_rss_gib
0,after_imports,0.517315,0.515732
1,after_orbits,0.520428,0.518784
2,after_waveform,6.557514,6.650955
3,after_grid,6.971981,7.297310
4,after_plan,7.155373,8.185623
5,after_dwt_benchmarks,7.787273,8.185623
6,after_wdm_cache,7.787655,8.185623
7,after_device_coefficients,7.787659,8.185623
8,after_materialized_shift_benchmark,8.335667,8.797764
9,after_schedule_sweep,9.395702,10.265133


Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_desktop_profile.json
Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_desktop_shift_schedule.csv
Saved: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/examples/_local_wdm_03_memory_stages.csv
